<a href="https://colab.research.google.com/github/diyamofficial/Internship_June2026/blob/main/Phase2_RAG_Optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STEP 0 — Install pinned, tested-compatible dependencies
# ============================================================
# These exact versions were verified together (clean install, pip check passes,
# all ragas/langchain/groq/gemini/chromadb imports succeed). Installing ragas
# or langchain-community "latest" instead of these pins is what causes the
# ModuleNotFoundError: langchain_community.chat_models.vertexai conflict.

!pip install -q \
  "ragas==0.2.15" \
  "langchain-community==0.3.27" \
  "langchain-groq==0.2.5" \
  "langchain-google-genai==2.0.11" \
  "chromadb==0.5.23" \
  "langgraph==0.2.74" \
  "pypdf" "docx2txt" "wikipedia"

print("Install complete.")
print("IMPORTANT: in Colab, go to Runtime -> Restart session now, then continue")
print("from the next cell down. This is required because chromadb/numpy/pyarrow")
print("ship compiled extensions that the old, already-imported versions in this")
print("kernel won't pick up without a restart -- skipping this step is the second")
print("most common source of 'works in pip, breaks at runtime' RAGAS errors in Colab.")


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.4/151.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
# ============================================================
# ALL IMPORTS
# ============================================================

# Basic Python
import os
import time
import json
import requests
import pandas as pd

# Wikipedia + Document Loading
import wikipedia
from langchain_core.documents import Document

# Text Splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ChromaDB
import chromadb

# Gemini Embeddings + Gemini LLM
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

# Groq (if needed)
from langchain_groq import ChatGroq

# LangGraph
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END

# RAGAS
from datasets import Dataset
from ragas import evaluate

from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

print("All imports loaded successfully!")

In [ ]:
# ============================================================
# STEP 0b — Imports and API keys
# ============================================================
import os, glob, re, json, time
from getpass import getpass

# Works whether you're in Colab or plain Jupyter
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass("Gemini API key: ")
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass("Groq API key: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Keys loaded.")


Gemini API key: ··········
Groq API key: ··········
Keys loaded.


In [ ]:
# ============================================================
# STEP 1a — Load your document set
# ============================================================
# Put 10-20 PDFs/DOCX files in this folder (upload them in Colab's file browser,
# or mount Drive). If you'd rather use Wikipedia articles, skip to STEP 1a-alt.

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

DOCS_FOLDER = "/content/source_docs"  # change to wherever your files are

def load_local_documents(folder_path):
    docs = []
    for path in glob.glob(os.path.join(folder_path, "*")):
        try:
            if path.lower().endswith(".pdf"):
                docs.extend(PyPDFLoader(path).load())
            elif path.lower().endswith(".docx"):
                docs.extend(Docx2txtLoader(path).load())
        except Exception as e:
            print(f"Skipped {path}: {e}")
    print(f"Loaded {len(docs)} pages/sections from {folder_path}")
    return docs

# raw_docs = load_local_documents(DOCS_FOLDER)


In [ ]:
# ============================================================
# STEP 1aalt — Load Knowledge Base Documents from Wikipedia
# ============================================================
# Fetches machine learning and AI-related articles from Wikipedia
# using the MediaWiki API. Includes rate-limit handling and stores
# each article as a LangChain Document with source metadata.
import time
import requests

def get_wikipedia_article(title, retries=5):
    headers = {
        "User-Agent": "MBCET-RAG-Project/1.0 (student project)"
    }

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "titles": title,
        "format": "json"
    }

    for attempt in range(retries):

        response = requests.get(
            url,
            params=params,
            headers=headers
        )

        if response.status_code == 429:
            wait = 5 * (attempt + 1)
            print(f"Rate limited. Waiting {wait}s...")
            time.sleep(wait)
            continue

        response.raise_for_status()

        data = response.json()

        pages = data["query"]["pages"]

        for page_id in pages:
            page = pages[page_id]

            if "extract" in page:
                return Document(
                    page_content=page["extract"],
                    metadata={
                        "source": page["title"]
                    }
                )

    return None

In [ ]:
TOPICS = [
    "Machine learning",
    "Deep learning",
    "Artificial neural network",
    "Convolutional neural network",
    "Recurrent neural network",
    "Long short-term memory",
    "Transformer (machine learning model)",
    "Large language model",
    "Backpropagation",
    "Gradient descent",
    "Computer vision",
    "Image classification",
    "Object detection",
    "Support vector machine",
    "Random forest",
    "Decision tree learning",
    "K-nearest neighbors algorithm",
    "Reinforcement learning"
]

raw_docs = []

for topic in TOPICS:
    try:
        doc = get_wikipedia_article(topic)

        if doc:
            raw_docs.append(doc)
            print("✓", topic)

    except Exception as e:
        print("✗", topic, e)

print("Loaded:", len(raw_docs))

✓ Machine learning
✓ Deep learning
✓ Artificial neural network
✓ Convolutional neural network
✓ Recurrent neural network
✓ Long short-term memory
✓ Transformer (machine learning model)
✓ Large language model
✓ Backpropagation
✓ Gradient descent
Rate limited. Waiting 5s...
✓ Computer vision
✓ Image classification
✓ Object detection
✓ Support vector machine
✓ Random forest
✓ Decision tree learning
✓ K-nearest neighbors algorithm
✓ Reinforcement learning
Loaded: 18


In [ ]:
# ============================================================
# STEP 1b — Sentence-aware 500-word chunking
# ============================================================

!pip install nltk -q

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize

def chunk_by_sentences(text, chunk_size=500, overlap=75):
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = []
    current_words = 0

    for sentence in sentences:
        sentence_words = len(sentence.split())

        if current_words + sentence_words > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))

            # Create overlap
            overlap_sentences = []
            overlap_words = 0

            for s in reversed(current_chunk):
                overlap_sentences.insert(0, s)
                overlap_words += len(s.split())

                if overlap_words >= overlap:
                    break

            current_chunk = overlap_sentences
            current_words = overlap_words

        current_chunk.append(sentence)
        current_words += sentence_words

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


all_chunks = []

for doc_id, doc in enumerate(raw_docs):
    pieces = chunk_by_sentences(
        doc.page_content,
        chunk_size=500,
        overlap=75
    )

    for i, piece in enumerate(pieces):
        all_chunks.append({
            "id": f"doc{doc_id}_chunk{i}",
            "text": piece,
            "metadata": {
                **doc.metadata,
                "chunk_index": i
            }
        })

print("Total chunks:", len(all_chunks))
print("\nSample chunk:\n")
print(all_chunks[0][:500] if isinstance(all_chunks[0], str) else all_chunks[0]["text"][:500])



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Total chunks: 232

Sample chunk:

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance. Statistics and mathematical optimisation methods compose the foundations o


In [ ]:
import os

print("DB folder exists:", os.path.exists("./chroma_db"))

DB folder exists: False


In [ ]:
# ============================================================
# STEP 1c — Embed each chunk with Gemini and store in ChromaDB
# (resumable + rate-limited + retries on quota errors)
# ============================================================
import time
import chromadb
from langchain_google_genai import GoogleGenerativeAIEmbeddings

EMBED_MODEL = "models/gemini-embedding-001"  # current model; text-embedding-004 is deprecated

embedder = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL)

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(
    name="rag_chunks",
    metadata={"hnsw:space": "cosine"},
)

def embed_batch_with_retry(texts, max_retries=5, base_delay=15):
    """Retries on quota/rate-limit errors with exponential backoff.
    langchain wraps the original google.api_core exception, so we have to
    detect the quota error by message content rather than exception type."""
    for attempt in range(max_retries):
        try:
            return embedder.embed_documents(texts, task_type="retrieval_document")
        except Exception as e:
            msg = str(e)
            if "RESOURCE_EXHAUSTED" in msg or "429" in msg or "quota" in msg.lower():
                wait = base_delay * (2 ** attempt)
                print(f"Quota/rate limit hit -- waiting {wait}s before retry {attempt + 1}/{max_retries}")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(
        "Still hitting quota errors after repeated retries with backoff. "
        "If this happens immediately on every retry, you've likely hit the FREE TIER "
        "DAILY quota (resets at midnight Pacific time), not a per-minute limit -- "
        "either wait for the reset or enable billing in AI Studio (Tier 1) to continue now. "
        "Re-running this same cell later will skip everything already embedded."
    )

def embed_and_store(chunks, batch_size=10, delay_between_batches=5):
    # Resume support: skip chunks already in the collection from a previous run
    existing_ids = set(collection.get(include=[])["ids"]) if collection.count() > 0 else set()
    pending = [c for c in chunks if c["id"] not in existing_ids]
    print(f"{len(existing_ids)} chunks already stored, {len(pending)} left to embed")

    for i in range(0, len(pending), batch_size):
        batch = pending[i:i + batch_size]
        texts = [c["text"] for c in batch]
        vectors = embed_batch_with_retry(texts)
        collection.add(
            ids=[c["id"] for c in batch],
            embeddings=vectors,
            documents=texts,
            metadatas=[c["metadata"] for c in batch],
        )
        print(f"Stored {i + len(batch)}/{len(pending)} remaining chunks")
        time.sleep(delay_between_batches)  # pacing to stay under free-tier RPM

embed_and_store(all_chunks)
print("Total chunks in ChromaDB:", collection.count())


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


0 chunks already stored, 232 left to embed


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored 10/232 remaining chunks
Stored 20/232 remaining chunks
Stored 30/232 remaining chunks
Stored 40/232 remaining chunks
Quota/rate limit hit -- waiting 15s before retry 1/5
Quota/rate limit hit -- waiting 30s before retry 2/5
Stored 50/232 remaining chunks
Stored 60/232 remaining chunks
Stored 70/232 remaining chunks
Stored 80/232 remaining chunks
Quota/rate limit hit -- waiting 15s before retry 1/5
Quota/rate limit hit -- waiting 30s before retry 2/5
Stored 90/232 remaining chunks
Stored 100/232 remaining chunks
Stored 110/232 remaining chunks
Quota/rate limit hit -- waiting 15s before retry 1/5
Quota/rate limit hit -- waiting 30s before retry 2/5
Stored 120/232 remaining chunks
Stored 130/232 remaining chunks
Quota/rate limit hit -- waiting 15s before retry 1/5
Quota/rate limit hit -- waiting 30s before retry 2/5
Stored 140/232 remaining chunks
Stored 150/232 remaining chunks
Stored 160/232 remaining chunks
Quota/rate limit hit -- waiting 15s before retry 1/5
Quota/rate limit hit

## Step 2 — Retrieval: top-5 most relevant chunks for any question

In [ ]:
!zip -r chroma_db.zip chroma_db

  adding: chroma_db/ (stored 0%)
  adding: chroma_db/chroma.sqlite3 (deflated 53%)
  adding: chroma_db/a71dafbb-b49b-4034-b620-7f32e9caa7f4/ (stored 0%)
  adding: chroma_db/a71dafbb-b49b-4034-b620-7f32e9caa7f4/length.bin (deflated 7%)
  adding: chroma_db/a71dafbb-b49b-4034-b620-7f32e9caa7f4/header.bin (deflated 61%)
  adding: chroma_db/a71dafbb-b49b-4034-b620-7f32e9caa7f4/data_level0.bin (deflated 100%)
  adding: chroma_db/a71dafbb-b49b-4034-b620-7f32e9caa7f4/link_lists.bin (stored 0%)


In [ ]:
from google.colab import files
files.download("chroma_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("rag_chunks")

print("Chunks loaded:", collection.count())

In [ ]:
# ============================================================
# STEP 2 — Retrieve the 5 most similar chunks for a question
# ============================================================
def retrieve(question, k=5):
    query_vector = embedder.embed_query(question)

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k
    )

    return [
        {
            "text": doc,
            "metadata": meta,
            "distance": dist
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ]


test_results = retrieve("What is reinforcement learning in machine learning?")

for r in test_results:
    print("="*60)
    print("Source:", r["metadata"]["source"])
    print("Distance:", round(r["distance"], 3))
    print(r["text"][:300])


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Source: Reinforcement learning
Distance: 0.252
In machine learning and optimal control, reinforcement learning (RL) is concerned with how an intelligent agent should take actions in a dynamic environment in order to maximize a reward signal. Reinforcement learning is one of the three basic machine learning paradigms, alongside supervised learnin
Source: Reinforcement learning
Distance: 0.26
R
          
            a
          
        
        (
        s
        ,
        
          s
          ′
        
        )
      
    
    {\displaystyle R_{a}(s,s')}
  
, the immediate reward after transitioning from 
  
    
      
        s
      
    
    {\displaystyle s}
  
 to 
  
    

Source: Machine learning
Distance: 0.279
One of the popular methods of dimensionality reduction is principal component analysis (PCA). PCA involves changing higher-dimensional data (e.g., 3D) to a smaller space (e.g., 2D). The manifold hypothesis proposes that high-dimensional data sets lie along low-dim

## Step 3 — Generation: pass retrieved chunks to Groq with a grounded prompt



In [ ]:
# ============================================================
# STEP 3 — Generate an answer with Groq using retrieved chunks
# ============================================================

from langchain_groq import ChatGroq

# Initialize Groq LLM
GROQ_MODEL = "llama-3.3-70b-versatile"

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.1
)

PROMPT_TEMPLATE = """
You are a helpful AI assistant.

Use ONLY the information provided in the context below.

If the answer cannot be found in the context, respond:

"I don't have enough information in the provided context to answer that."

Context:
{context}

Question:
{question}

Answer:
"""

def generate_answer(question, retrieved_chunks):

    context = "\n\n".join(
        f"[{i+1}] {chunk['text']}"
        for i, chunk in enumerate(retrieved_chunks)
    )

    prompt = PROMPT_TEMPLATE.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    return response.content

# ============================================================
# INTERACTIVE TEST LOOP
# ============================================================

while True:
    question = input("Ask a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    retrieved_chunks = retrieve(question, k=5)

    answer = generate_answer(
        question,
        retrieved_chunks
    )

    print("\n" + "="*60)
    print(answer)
    print("="*60 + "\n")

Ask a question (type 'exit' to quit): why deep learning used for image analysis

Deep learning is used for image analysis because it can learn to recognize patterns and features in images, allowing it to perform tasks such as image recognition, object detection, and image segmentation with high accuracy. Deep learning models, such as convolutional neural networks (CNNs), can automatically learn to extract relevant features from images, eliminating the need for manual feature engineering. This makes deep learning particularly well-suited for image analysis tasks, where the number of possible features and patterns can be very large. Additionally, deep learning models can learn to recognize complex patterns and relationships in images, allowing them to perform tasks such as image classification, object detection, and image generation.

Ask a question (type 'exit' to quit): difference between ANN and CNN

The main difference between Artificial Neural Networks (ANN) and Convolutional Neural

## Step 4 — Wire it together as a LangGraph pipeline with real error handling

This is the part the brief explicitly calls out ("proper error handling") and the part that's easiest to skip. The graph below has three things a plain function call doesn't give you for free: a **retry** around the Groq call (transient API errors, rate limits), a **fallback path when retrieval comes back empty** (instead of asking the LLM to hallucinate an answer from nothing), and a **typed state object** so each node's inputs/outputs are explicit and inspectable.

In [ ]:
from langchain_groq import ChatGroq

GROQ_MODEL = "llama-3.3-70b-versatile"

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.1
)


PROMPT_TEMPLATE = """
You are a helpful AI assistant.

Use ONLY the information provided in the context below.

If the answer cannot be found in the context, respond:

"I don't have enough information in the provided context to answer that."

Context:
{context}

Question:
{question}

Answer:
"""

def generate_from_chunks(question, retrieved_chunks):

    context = "\n\n".join(
        f"[{i+1}] {chunk['text']}"
        for i, chunk in enumerate(retrieved_chunks)
    )

    prompt = PROMPT_TEMPLATE.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    return response.content

In [ ]:
chunks = retrieve("What is machine learning?")

answer = generate_from_chunks(
    "What is machine learning?",
    chunks
)

print(answer)

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


In [ ]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END
import time

class RAGState(TypedDict):
    question: str
    retrieved_chunks: Optional[List[dict]]
    answer: Optional[str]
    error: Optional[str]


def retrieve_node(state: RAGState) -> RAGState:

    try:
        results = retrieve(state["question"], k=5)

        if not results:
            return {
                **state,
                "retrieved_chunks": [],
                "error": "no_results"
            }

        return {
            **state,
            "retrieved_chunks": results,
            "error": None
        }

    except Exception as e:

        return {
            **state,
            "retrieved_chunks": None,
            "error": f"retrieval_failed: {e}"
        }


def generate_node(state: RAGState) -> RAGState:

    if state.get("error") == "no_results":

        return {
            **state,
            "answer": "I couldn't find any relevant information in the document set to answer this question."
        }

    if state.get("error"):

        return {
            **state,
            "answer": f"Pipeline error before generation: {state['error']}"
        }

    max_retries = 3
    last_exception = None

    for attempt in range(max_retries):

        try:

            answer = generate_from_chunks(
                state["question"],
                state["retrieved_chunks"]
            )

            return {
                **state,
                "answer": answer,
                "error": None
            }

        except Exception as e:

            last_exception = e

            time.sleep(2 ** attempt)

    return {
        **state,
        "answer": None,
        "error": f"generation_failed after {max_retries} retries: {last_exception}"
    }


graph = StateGraph(RAGState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)

graph.set_entry_point("retrieve")

graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

rag_pipeline = graph.compile()

In [ ]:
result = rag_pipeline.invoke({
    "question": "what is machine learning",
    "retrieved_chunks": None,
    "answer": None,
    "error": None
})

print(result["answer"])

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


## Step 5 — Evaluate with RAGAS



In [ ]:
from datasets import Dataset

# ============================================================
# 20 Academic-Level Evaluation Questions
# ============================================================

test_questions = [
    "How does deep learning differ from traditional machine learning approaches?",
    "Why do deep neural networks outperform many classical machine learning methods on complex tasks?",
    "How does backpropagation enable learning in neural networks?",
    "Explain the relationship between gradient descent and backpropagation during training.",
    "How does an LSTM mitigate the vanishing gradient problem found in RNNs?",
    "Why are transformer architectures preferred over recurrent neural networks for large language models?",
    "How does self-attention improve sequence modeling compared to recurrence?",
    "What role do embeddings play in large language models?",
    "How does reinforcement learning differ from supervised learning in terms of feedback mechanisms?",
    "Why is exploration-exploitation tradeoff important in reinforcement learning?",
    "Compare image classification and object detection tasks in computer vision.",
    "How do convolutional neural networks extract hierarchical visual features?",
    "What advantages does a Random Forest provide over a single Decision Tree?",
    "How does ensemble learning improve predictive performance?",
    "What is the significance of support vectors in Support Vector Machines?",
    "How does the kernel trick extend the capability of SVMs?",
    "How does the K-Nearest Neighbors algorithm perform classification?",
    "Why is feature representation important for machine learning performance?",
    "What factors contribute to overfitting in machine learning models?",
    "Compare the learning mechanisms of supervised, unsupervised, and reinforcement learning."
]

ground_truths = [
    "Deep learning uses multi-layer neural networks to automatically learn representations, while traditional machine learning often relies on manually engineered features.",
    "Deep neural networks learn hierarchical feature representations that enable superior performance on complex tasks such as vision and language processing.",
    "Backpropagation computes gradients of the loss function with respect to network parameters and propagates errors backward through the network.",
    "Backpropagation calculates gradients while gradient descent uses those gradients to update model parameters and minimize loss.",
    "LSTMs use memory cells and gating mechanisms to preserve information over long sequences and reduce vanishing gradients.",
    "Transformers process sequences in parallel and use self-attention, enabling better scalability and long-range dependency modeling.",
    "Self-attention allows each token to attend directly to all other relevant tokens regardless of distance in the sequence.",
    "Embeddings convert discrete tokens into dense vector representations that capture semantic relationships.",
    "Supervised learning learns from labeled examples, while reinforcement learning learns through rewards obtained from interactions with an environment.",
    "The exploration-exploitation tradeoff balances discovering new actions against using actions already known to provide rewards.",
    "Image classification predicts labels for an entire image, whereas object detection identifies and localizes objects within an image.",
    "CNNs use convolutional filters to learn increasingly abstract features through successive layers.",
    "Random Forest combines multiple decision trees to reduce variance and improve generalization.",
    "Ensemble learning combines multiple models to improve robustness and predictive accuracy.",
    "Support vectors are the critical training samples closest to the decision boundary that determine the separating hyperplane.",
    "The kernel trick enables SVMs to solve nonlinear problems by mapping data into higher-dimensional feature spaces.",
    "KNN classifies instances based on the labels of the nearest neighboring examples.",
    "Feature representation determines how effectively useful patterns can be captured by learning algorithms.",
    "Overfitting occurs when models learn noise and dataset-specific patterns rather than generalizable relationships.",
    "Supervised learning uses labeled data, unsupervised learning discovers structure in unlabeled data, and reinforcement learning learns through reward-driven interaction."
]

# ============================================================
# Build Evaluation Dataset
# ============================================================

from datasets import Dataset

data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for q, gt in zip(test_questions, ground_truths):

    retrieved = retrieve(q, k=3)

    answer = generate_answer(q, retrieved)

    contexts = [
        chunk["text"][:1000]
        for chunk in retrieved
    ]

    data["question"].append(q)
    data["answer"].append(answer)
    data["contexts"].append(contexts)
    data["ground_truth"].append(gt)

dataset = Dataset.from_dict(data)

print("Dataset created successfully")
print(dataset[0])

Dataset created successfully
{'question': 'How does deep learning differ from traditional machine learning approaches?', 'answer': 'Deep learning differs from traditional machine learning approaches in that it does not require hand-crafted feature engineering. Instead, the model discovers useful feature representations from the data automatically through the use of multiple layers. This allows the model to learn and transform the input data into a progressively more abstract and composite representation. In traditional machine learning, features are often hand-crafted and then fed into a classification algorithm, whereas in deep learning, the model learns to extract relevant features from the data on its own.', 'contexts': ['In machine learning, deep learning (DL) focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. The field takes inspiration from biological neuroscience and revolves around stacking artific

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [ ]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from datasets import Dataset
import pandas as pd
import time

In [ ]:
eval_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

eval_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

ragas_llm = LangchainLLMWrapper(eval_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(eval_embeddings)

In [ ]:
all_results = []

for i in range(len(data["question"])):

    print(f"\nEvaluating {i+1}/{len(data['question'])}")

    sample = Dataset.from_dict({
        "question": [data["question"][i]],
        "answer": [data["answer"][i]],
        "contexts": [data["contexts"][i]],
        "ground_truth": [data["ground_truth"][i]]
    })

    try:

        result = evaluate(
            dataset=sample,
            metrics=[
                Faithfulness(),
                AnswerRelevancy(),
                ContextPrecision(),
                ContextRecall()
            ],
            llm=ragas_llm,
            embeddings=ragas_embeddings
        )

        row = result.to_pandas().iloc[0]

        all_results.append({
            "Question": data["question"][i],
            "Generated Answer": data["answer"][i],
            "Faithfulness": row.get("faithfulness"),
            "Answer Relevancy": row.get("answer_relevancy"),
            "Context Precision": row.get("context_precision"),
            "Context Recall": row.get("context_recall")
        })

        print("✓ Success")

        time.sleep(15)

    except Exception as e:

        print("Failed:", e)

        all_results.append({
            "Question": data["question"][i],
            "Generated Answer": data["answer"][i],
            "Faithfulness": None,
            "Answer Relevancy": None,
            "Context Precision": None,
            "Context Recall": None
        })

        time.sleep(30)


Evaluating 1/20


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Success

Evaluating 2/20


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Success

Evaluating 3/20


...

✓ Success

Evaluating 19/20


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Success

Evaluating 20/20


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Success


In [ ]:
final_df = pd.DataFrame(all_results)

display(final_df)

,Question,Generated Answer,Faithfulness,Answer Relevancy,Context Precision,Context Recall
0,How does deep learning differ from traditional...,Deep learning differs from traditional machine...,0.714286,0.844425,1.000000,1.000000
1,Why do deep neural networks outperform many cl...,Deep neural networks outperform many classical...,0.333333,0.854263,1.000000,0.500000
2,How does backpropagation enable learning in ne...,Backpropagation enables learning in neural net...,0.714286,0.840043,0.833333,1.000000
3,Explain the relationship between gradient desc...,Gradient descent is an algorithm used to find ...,1.000000,0.762366,1.000000,1.000000
4,How does an LSTM mitigate the vanishing gradie...,An LSTM mitigates the vanishing gradient probl...,0.500000,0.865965,0.833333,1.000000
5,Why are transformer architectures preferred ov...,Transformer architectures are preferred over r...,0.166667,0.879924,1.000000,0.000000
6,How does self-attention improve sequence model...,I don't have enough information in the provide...,0.000000,0.000000,1.000000,0.000000
7,What role do embeddings play in large language...,"Embeddings, such as word embeddings (e.g., Wor...",0.666667,0.886314,1.000000,0.000000
8,How does reinforcement learning differ from su...,"In reinforcement learning, the agent receives ...",1.000000,0.794415,1.000000,0.500000
9,Why is exploration-exploitation tradeoff impor...,The exploration-exploitation tradeoff is impor...,1.000000,0.824597,1.000000,0.000000


In [ ]:
avg_df = pd.DataFrame({
    "Metric":[
        "Faithfulness",
        "Answer Relevancy",
        "Context Precision",
        "Context Recall"
    ],
    "Average Score":[
        final_df["Faithfulness"].mean(),
        final_df["Answer Relevancy"].mean(),
        final_df["Context Precision"].mean(),
        final_df["Context Recall"].mean()
    ]
})

display(avg_df)

,Metric,Average Score
0,Faithfulness,0.622253
1,Answer Relevancy,0.754734
2,Context Precision,0.895833
3,Context Recall,0.408333


In [ ]:
worst_df = final_df.sort_values(
    by=["Answer Relevancy","Faithfulness"]
).head(3)

display(worst_df)

,Question,Generated Answer,Faithfulness,Answer Relevancy,Context Precision,Context Recall
6,How does self-attention improve sequence model...,I don't have enough information in the provide...,0.0,0.000000,1.0,0.0
10,Compare image classification and object detect...,I don't have enough information in the provide...,0.0,0.000000,1.0,0.0
3,Explain the relationship between gradient desc...,Gradient descent is an algorithm used to find ...,1.0,0.762366,1.0,1.0


In [ ]:
final_df.to_csv("ragas_results.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


In [ ]:
from google.colab import files

files.download("ragas_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>